In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
# =========================================================
# Symptom2Disease - 50 Experiments (5 models × 5 feature sizes × 2 TF-IDF types)
# =========================================================

import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings("ignore")

# ----------------------------
# Load dataset
# ----------------------------
df = pd.read_csv('/content/drive/MyDrive/cleaned_encoded.csv')
X = df['cleaned_text']
y = df['encoded_label']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ----------------------------
# Define models + params
# ----------------------------
models = {
    "NaiveBayes": {
        "model": MultinomialNB(),
        "params": {"alpha": [0.5, 1.0]}
    },
    "SVM": {
        "model": SVC(),
        "params": {"C": [1, 10], "kernel": ["linear"]}
    },
    "LogisticRegression": {
        "model": LogisticRegression(max_iter=500),
        "params": {"C": [1, 10]}
    },
    "RandomForest": {
        "model": RandomForestClassifier(),
        "params": {"n_estimators": [100], "max_depth": [None, 10]}
    },
    "XGBoost": {
        "model": XGBClassifier(eval_metric="mlogloss", use_label_encoder=False),
        "params": {"learning_rate": [0.05, 0.1], "max_depth": [3, 5], "n_estimators": [100]}
    }
}

# ----------------------------
# TF-IDF feature sizes
# ----------------------------
feature_sizes = [2000, 3000, 3500, 4000, 4500]

# ----------------------------
# Run experiments
# ----------------------------
experiment_results = []

# ----- Word Unigrams (25 experiments) -----
for max_feat in feature_sizes:
    vectorizer = TfidfVectorizer(max_features=max_feat, ngram_range=(1,1), stop_words='english')
    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_test_tfidf = vectorizer.transform(X_test)

    for name, mp in models.items():
        print(f"Word-Unigram | {name} | Max Features: {max_feat}")
        grid = GridSearchCV(mp['model'], mp['params'], cv=3, n_jobs=-1)
        grid.fit(X_train_tfidf, y_train)
        best_model = grid.best_estimator_
        y_pred = best_model.predict(X_test_tfidf)
        acc = accuracy_score(y_test, y_pred)

        experiment_results.append({
            "Model": name,
            "TFIDF_Type": "word-unigram",
            "Max_Features": max_feat,
            "Best_Params": grid.best_params_,
            "Accuracy": acc
        })

# ----- Char 7-grams (25 experiments) -----
for max_feat in feature_sizes:
    vectorizer = TfidfVectorizer(max_features=max_feat, analyzer='char', ngram_range=(7,7))
    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_test_tfidf = vectorizer.transform(X_test)

    for name, mp in models.items():
        print(f"Char-7gram | {name} | Max Features: {max_feat}")
        grid = GridSearchCV(mp['model'], mp['params'], cv=3, n_jobs=-1)
        grid.fit(X_train_tfidf, y_train)
        best_model = grid.best_estimator_
        y_pred = best_model.predict(X_test_tfidf)
        acc = accuracy_score(y_test, y_pred)

        experiment_results.append({
            "Model": name,
            "TFIDF_Type": "char-7gram",
            "Max_Features": max_feat,
            "Best_Params": grid.best_params_,
            "Accuracy": acc
        })

# ----------------------------
# Save results
# ----------------------------
results_df = pd.DataFrame(experiment_results)
results_df.to_csv("experiment_results.csv", index=False)
print("\n✅ 50 experiments completed! Results saved to 'experiment_results.csv'.")

# ----------------------------
# Best model
# ----------------------------
best_row = results_df.loc[results_df['Accuracy'].idxmax()]
print("\n✅ Best model configuration:")
print(best_row)


Word-Unigram | NaiveBayes | Max Features: 2000
Word-Unigram | SVM | Max Features: 2000
Word-Unigram | LogisticRegression | Max Features: 2000
Word-Unigram | RandomForest | Max Features: 2000
Word-Unigram | XGBoost | Max Features: 2000
Word-Unigram | NaiveBayes | Max Features: 3000
Word-Unigram | SVM | Max Features: 3000
Word-Unigram | LogisticRegression | Max Features: 3000
Word-Unigram | RandomForest | Max Features: 3000
Word-Unigram | XGBoost | Max Features: 3000
Word-Unigram | NaiveBayes | Max Features: 3500
Word-Unigram | SVM | Max Features: 3500
Word-Unigram | LogisticRegression | Max Features: 3500
Word-Unigram | RandomForest | Max Features: 3500
Word-Unigram | XGBoost | Max Features: 3500
Word-Unigram | NaiveBayes | Max Features: 4000
Word-Unigram | SVM | Max Features: 4000
Word-Unigram | LogisticRegression | Max Features: 4000
Word-Unigram | RandomForest | Max Features: 4000
Word-Unigram | XGBoost | Max Features: 4000
Word-Unigram | NaiveBayes | Max Features: 4500
Word-Unigram 

In [1]:
import pandas as pd

# Load the results if they are not already in memory
# (Assuming experiment_results.csv is in the current working directory or specify the full path)
try:
    results_df = pd.read_csv('experiment_results.csv')
except NameError:
    print("results_df not found in memory. Please ensure the previous cell that generates it has been run.")
except FileNotFoundError:
    print("experiment_results.csv not found. Please ensure it was created and is in the correct directory.")

# Set display options to show all rows and columns
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.colheader_justify', 'left')
pd.set_option('display.expand_frame_repr', False)

# Display the full DataFrame
print("\nFull Experiment Results:")
display(results_df)

# Display the best model configuration again for clarity
print("\nBest model configuration:")
display(results_df.loc[results_df['Accuracy'].idxmax()])

# Reset display options to default after displaying
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.width')
pd.reset_option('display.colheader_justify')
pd.reset_option('display.expand_frame_repr')



Full Experiment Results:


,Model,TFIDF_Type,Max_Features,Best_Params,Accuracy
0,NaiveBayes,word-unigram,2000,{'alpha': 0.5},0.943723
1,SVM,word-unigram,2000,"{'C': 10, 'kernel': 'linear'}",0.982684
2,LogisticRegression,word-unigram,2000,{'C': 10},0.969697
3,RandomForest,word-unigram,2000,"{'max_depth': None, 'n_estimators': 100}",0.943723
4,XGBoost,word-unigram,2000,"{'learning_rate': 0.1, 'max_depth': 3, 'n_esti...",0.887446
5,NaiveBayes,word-unigram,3000,{'alpha': 0.5},0.943723
6,SVM,word-unigram,3000,"{'C': 10, 'kernel': 'linear'}",0.982684
7,LogisticRegression,word-unigram,3000,{'C': 10},0.969697
8,RandomForest,word-unigram,3000,"{'max_depth': None, 'n_estimators': 100}",0.939394
9,XGBoost,word-unigram,3000,"{'learning_rate': 0.1, 'max_depth': 3, 'n_esti...",0.887446



Best model configuration:


Model                                     SVM
TFIDF_Type                       word-unigram
Max_Features                             2000
Best_Params     {'C': 10, 'kernel': 'linear'}
Accuracy                             0.982684
Name: 1, dtype: object